# NutriScan — fine-tune bake-off (Colab GPU)

Runs `ml/train_finetune.py` on a GPU for both backbones (EfficientNetV2-S and ViT-S), then compares them against the Day-6 baseline on the frozen test set.

**Before you start:**
1. Locally run `uv run python ml/pack_dataset.py` → produces `ml/data/nutriscan_dataset_v1.zip`.
2. Upload that zip to Google Drive at `MyDrive/nutriscan/nutriscan_dataset_v1.zip`.
3. Here in Colab: **Runtime → Change runtime type → GPU** (T4 is fine).
4. Run the cells top to bottom.

The baseline to beat is **Top-1 86.1% / Top-5 97.1%**.

In [ ]:
# Confirm a GPU is attached — fail fast so we never silently train on CPU.
!nvidia-smi -L
import torch

assert torch.cuda.is_available(), "No CUDA GPU. Runtime -> Change runtime type -> GPU, then rerun."
print("CUDA device:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Paths — edit DATA_ZIP if you put the zip somewhere else in Drive.
DATA_ZIP = "/content/drive/MyDrive/nutriscan/nutriscan_dataset_v1.zip"
REPO = "https://github.com/Shivamchaubey14/nutriscan.git"
REPO_DIR = "/content/nutriscan"
OUT_DIR = "/content/drive/MyDrive/nutriscan/results"  # winners are copied back here
REV = ""  # optional: pin to the commit SHA that matches your dataset zip (blank = latest main)

In [ ]:
# Clean clone (avoid a stale /content/nutriscan) — brings code + committed manifests.
!rm -rf {REPO_DIR}
!git clone {REPO} {REPO_DIR}
%cd {REPO_DIR}
if REV:
    !git checkout {REV}
# Record exactly which commit trained this model (pairs with the dataset zip).
!git rev-parse HEAD

In [ ]:
# Unzip into a clean ml/data/raw/, then verify every manifest image is present.
import csv
import os
import shutil
import subprocess

assert os.path.exists(DATA_ZIP), f"dataset zip not found: {DATA_ZIP}"
shutil.rmtree("ml/data/raw", ignore_errors=True)  # no stale files from a previous run
os.makedirs("ml/data/raw", exist_ok=True)
subprocess.run(["unzip", "-q", DATA_ZIP, "-d", "ml/data/raw"], check=True)

refs = set()
for split in ("train", "val", "test"):
    with open(f"ml/manifests/{split}_v1.csv") as f:
        refs.update(r["path"] for r in csv.DictReader(f))
missing = [p for p in refs if not os.path.exists(os.path.join("ml/data/raw", p))]
assert not missing, f"{len(missing)} manifest images missing from the zip (first 3: {missing[:3]})"
print(f"dataset OK: {len(refs)} images present")

In [ ]:
# Colab already has CUDA torch/torchvision — just add timm + mlflow.
!pip -q install 'timm>=1.0.28' 'mlflow>=3.14'

In [ ]:
# Train EfficientNetV2-S (full fine-tune). ~15 epochs; adjust to taste.
!python ml/train_finetune.py --backbone effv2s --epochs 15 --batch-size 64 --num-workers 2

In [ ]:
# Train ViT-S on the same data + protocol.
!python ml/train_finetune.py --backbone vits --epochs 15 --batch-size 64 --num-workers 2

In [ ]:
# Compare candidates. Selection is by VALIDATION top-1 — the frozen test is a
# one-shot final number, never used to pick between models.
import json
import math
import os

candidates = ["baseline_v1", "effv2s_v1", "vits_v1"]
rows = []
for name in candidates:
    path = f"ml/metrics/{name}.json"
    if not os.path.exists(path):
        continue
    with open(path) as f:
        m = json.load(f)
    rows.append((m["model"], m.get("val_top1_best"), m.get("test_top1"), m.get("test_top5")))

print(f"{'model':<45}{'val_top1':>10}{'test_top1':>10}{'test_top5':>10}")
for name, val, t1, t5 in rows:
    print(f"{name:<45}{(val or math.nan):>10.4f}{(t1 or math.nan):>10.4f}{(t5 or math.nan):>10.4f}")

selectable = [r for r in rows if r[1] is not None]
if selectable:
    winner = max(selectable, key=lambda r: r[1])
    print(f"\nwinner by val_top1: {winner[0]}")
    print(
        f"its frozen-test top1 {winner[2]:.4f} / top5 {winner[3]:.4f} is the final number to report"
    )
print("baseline (frozen test) to beat: top1 0.8606 / top5 0.9707")

In [ ]:
# Copy the two trained candidates back to Drive (assert they exist first).
import os
import shutil

os.makedirs(OUT_DIR, exist_ok=True)
for name in ("effv2s", "vits"):
    for sub, ext in (("models", "pt"), ("metrics", "json")):
        path = f"ml/{sub}/{name}_v1.{ext}"
        assert os.path.exists(path), f"missing expected output: {path}"
        shutil.copy2(path, OUT_DIR)
print("copied to", OUT_DIR)
!ls -la {OUT_DIR}

## Next (Day 20 wrap-up, done locally)

1. Download the winning `<backbone>_v1.pt` + `<backbone>_v1.json` from `MyDrive/nutriscan/results` into `ml/models/` and `ml/metrics/`.
2. Commit the metrics JSON (the `.pt` stays git-ignored).
3. If the winner beats the 86.1% baseline, export it to ONNX (Day-8 style: `inference/export_onnx.py`) and wire it into the inference service.

MLflow runs are under `mlruns/` in the Colab session — download it too if you want the full history.